In [0]:
%run ./libraries

In [0]:
%run ./connection

In [0]:
# Use Snowflake data for analysis
# Replace synthetic data with df_sales_order

df_general_use = df_sales_order

display(df_general_use)

In [0]:
# Display Dataframe Schema Structure for Snowflake Data
df_general_use.printSchema()

In [0]:
# Count Total Entries in Sales Order Data
df_general_use.count()

In [0]:
# Clean and Display Dataframe Without Missing Values from Snowflake
df_general_use = df_general_use.dropna()
display(df_general_use)

In [0]:
# Convert Order_Date to Date Column (assuming column exists in Sales Order)
from pyspark.sql.functions import col, to_date

df_general_use = df_general_use.withColumn(
    "Order_Date",
    to_date(col("Order_Date"))
)
display(df_general_use)

In [0]:
# Aggregate and Display Daily Sales Order Quantities
from pyspark.sql.functions import sum

df_daily = df_general_use.groupBy("Order_Date").agg(
    sum("Quantity").alias("Quantity")
)
display(df_daily)

In [0]:
# Add Week, Month, and Day Features to df_daily
from pyspark.sql.functions import dayofweek, month, weekofyear

df_features = df_daily \
    .withColumn("day_of_week", dayofweek("Order_Date")) \
    .withColumn("month", month("Order_Date")) \
    .withColumn("week_of_year", weekofyear("Order_Date"))

display(df_features)

In [0]:
# Add Lag Features for Quantity Using Window Functions
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

windowSpec = Window.orderBy("Order_Date")

df_features = df_features \
    .withColumn("lag_1", lag("Quantity",1).over(windowSpec)) \
    .withColumn("lag_7", lag("Quantity",7).over(windowSpec))

display(df_features)

In [0]:
# Calculate 7-Day Rolling Average for Quantity Sold
from pyspark.sql.functions import avg
from pyspark.sql.window import Window

rollingWindow = Window.orderBy("Order_Date").rowsBetween(-7,0)
df_features = df_features.withColumn(
    "rolling_avg_7",
    avg("Quantity").over(rollingWindow)
)
display(df_features)

In [0]:
# Remove Missing Values and Display Cleaned Feature DataFrame
df_features = df_features.dropna()
display(df_features)

In [0]:
# Convert Features DataFrame to Pandas and Preview Data
pdf = df_features.toPandas()
pdf.head()

In [0]:
features = [
    "day_of_week",
    "month",
    "week_of_year",
    "lag_1",
    "lag_7",
    "rolling_avg_7"
]

X = pdf[features]
y = pdf["Quantity"]

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)